# Football Data API — Exploration

Objectivo: explorar os endpoints da [football-data.org](https://www.football-data.org/) para o Mundial 2026, mapear os dados para o nosso schema, e prototipar as funções de import.

**Competition code:** `WC` (FIFA World Cup)

In [ ]:
import os
from pprint import pprint

import requests

API_KEY = os.environ.get("FOOTBALL_DATA_API_KEY", "YOUR_KEY_HERE")
BASE_URL = "https://api.football-data.org/v4"
HEADERS = {"X-Auth-Token": API_KEY}


def get(endpoint: str, params: dict = None) -> dict:
    """Helper to call the API."""
    url = f"{BASE_URL}/{endpoint}"
    resp = requests.get(url, headers=HEADERS, params=params)
    resp.raise_for_status()
    return resp.json()


print(f"API Key loaded: {'yes' if API_KEY != 'YOUR_KEY_HERE' else 'NO — set FOOTBALL_DATA_API_KEY'}")

## 1. Competition Info

Verificar se o Mundial 2026 está disponível na API.

In [ ]:
competition = get("competitions/WC")
pprint(
    {
        "id": competition["id"],
        "name": competition["name"],
        "code": competition["code"],
        "currentSeason": competition.get("currentSeason"),
    }
)

## 2. Teams

Listar as equipas qualificadas. Mapear para o nosso schema:

| API field | Our field | Notes |
|-----------|-----------|-------|
| `name` | `name` | |
| `tla` | `code` | 3-letter code |
| `crest` | `flag_url` | |
| `?` | `group_letter` | Pode vir do endpoint de standings |
| `area.name` | `confederation` | Aproximação via área geográfica |

In [ ]:
teams_data = get("competitions/WC/teams")
print(f"Total teams: {teams_data['count']}")
print()

# Inspect first team structure
if teams_data["teams"]:
    pprint(teams_data["teams"][0])

In [ ]:
# List all teams with key fields
for team in teams_data["teams"]:
    print(f"{team['tla']:>4} | {team['name']:<30} | {team.get('area', {}).get('name', 'N/A')}")

## 3. Groups / Standings

Obter a distribuição por grupos (se disponível).

In [ ]:
try:
    standings = get("competitions/WC/standings")
    for group in standings.get("standings", []):
        print(f"\n--- {group['group']} ---")
        for entry in group["table"]:
            print(f"  {entry['team']['tla']} - {entry['team']['name']}")
except Exception as e:
    print(f"Standings not available yet: {e}")

## 4. Matches / Fixtures

Mapear para o nosso schema:

| API field | Our field | Notes |
|-----------|-----------|-------|
| `homeTeam.tla` | `home_team_id` | Lookup by code |
| `awayTeam.tla` | `away_team_id` | Lookup by code |
| `stage` | `stage` | Needs mapping |
| `group` | `group_letter` | Extract letter |
| `matchday` | `match_number` | |
| `utcDate` | `match_date` | ISO format |
| `venue` | `venue` | |
| `status` | `status` | Needs mapping |

In [ ]:
matches_data = get("competitions/WC/matches")
print(f"Total matches: {matches_data['resultSet']['count']}")
print()

# Inspect first match structure
if matches_data["matches"]:
    pprint(matches_data["matches"][0])

In [ ]:
# List matches with key info
for m in matches_data["matches"][:10]:  # first 10
    home = m.get("homeTeam", {}).get("tla", "TBD")
    away = m.get("awayTeam", {}).get("tla", "TBD")
    print(f"Match {m['matchday']:>2} | {m['stage']:<20} | {home} vs {away} | {m['utcDate'][:10]} | {m['status']}")

## 5. Squads / Players

Obter os jogadores de cada equipa.

In [ ]:
# Pick first team to explore squad structure
if teams_data["teams"]:
    first_team = teams_data["teams"][0]
    team_id = first_team["id"]
    print(f"Exploring squad for: {first_team['name']} (id={team_id})")
    print()

    team_detail = get(f"teams/{team_id}")
    squad = team_detail.get("squad", [])
    print(f"Squad size: {len(squad)}")

    if squad:
        pprint(squad[0])  # inspect one player
        print()
        for p in squad[:5]:
            print(f"  {p.get('position', 'N/A'):<12} | {p['name']}")

## 6. Field Mapping Summary

Depois de explorar, documentar o mapeamento final aqui.

### Stage mapping
```python
STAGE_MAP = {
    "GROUP_STAGE": "group",
    "ROUND_OF_32": "R32",  # new for 2026 (48 teams)
    "LAST_16": "R16",
    "QUARTER_FINALS": "QF",
    "SEMI_FINALS": "SF",
    "THIRD_PLACE": "3rd",
    "FINAL": "F",
}
```

### Position mapping
```python
POSITION_MAP = {
    "Goalkeeper": "GK",
    "Defence": "DF",
    "Midfield": "MF",
    "Offence": "FW",
}
```

## Next Steps

1. Confirmar quais campos estão disponíveis para WC 2026
2. Identificar edge cases (equipas sem grupo, jogos TBD, etc.)
3. Prototipar funções de upsert
4. Extrair para `pipelines/` como módulos ETL (extract → transform → load)